In [ ]:
import pandas as pd
import folium
import math

# 1. CARGAR TU BASE DE DATOS EXACTA
df = pd.read_csv('datos_cuatro_comunas.csv')

# Filtrar solo el modo BUS y pasar a mayúsculas para que cruce perfecto
df_buses = df[df['Modo'].str.contains('BUS', case=False, na=False)].copy()
df_buses['Comuna'] = df_buses['Comuna'].str.upper()

# 2. AGRUPAR LOS DATOS (Sin cortes de AM/PM, sumando el volumen total)
df_buses['Subidas_Promedio'] = pd.to_numeric(df_buses['Subidas_Promedio'], errors='coerce').fillna(0)
df_agrupado = df_buses.groupby('Comuna')['Subidas_Promedio'].sum().reset_index()

# 3. TUS 4 COMUNAS EXACTAS Y SUS COORDENADAS
coordenadas_comunas = {
    'SANTIAGO': {'coord': [-33.4372, -70.6506], 'color': '#007BFF'},     # Azul
    'MAIPÚ': {'coord': [-33.5166, -70.7666], 'color': '#ff7f0e'},        # Naranja
    'LAS CONDES': {'coord': [-33.4166, -70.5833], 'color': '#2ca02c'},   # Verde
    'PUENTE ALTO': {'coord': [-33.6166, -70.5833], 'color': '#d62728'}   # Rojo
}

# 4. CREAR EL MAPA ÚNICO BASE (Fondo oscuro, centrado en Santiago)
mapa = folium.Map(location=[-33.4489, -70.6693], zoom_start=11, tiles='CartoDB dark_matter')

# 5. DIBUJAR LOS GLOBOS DE CALOR
for index, row in df_agrupado.iterrows():
    comuna = row['Comuna']
    flujo_total = row['Subidas_Promedio']
    
    if comuna in coordenadas_comunas and flujo_total > 0:
        lat, lon = coordenadas_comunas[comuna]['coord']
        color = coordenadas_comunas[comuna]['color']
        
        # Fórmula para que la burbuja crezca según el total de pasajeros
        radio = max(5, math.sqrt(flujo_total) * 0.08)
        
        folium.CircleMarker(
            location=[lat, lon],
            radius=radio,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.75,
            weight=2,
            tooltip=f"<div style='font-family: Arial, sans-serif; font-size: 13px;'>"
                    f"<b>Comuna:</b> {comuna}<br>"
                    f"<b>Pasajeros Totales (Buses):</b> {int(flujo_total):,}</div>"
        ).add_to(mapa)

# 6. EXPORTAR EL ARCHIVO FINAL
mapa.save('mapa_4comunas_original.html')
print("✓ Mapa original guardado exitosamente como mapa_4comunas_original.html")